In [ ]:
import os
import numpy as np
import pandas as pd
from gudhi.representations import Silhouette
from scipy.ndimage import gaussian_filter1d

sigma = 2
resolution = 100
normalization = "none"
THR_MODE = "p10"

base = "RIPS"
dim_intervals = [0, 1]


def read_and_save(filedir, tube):
    if tube and tube[0] != ".":
        file_name, file_extension = os.path.splitext(os.path.join(filedir, tube))
        tubenamerips = tube.split("_")[-1].split(".")[0]
        if file_extension != ".pdf" and tubenamerips == "Rips0":
            r0_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips0.txt")
            r1_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips1.txt")

            Rips0 = np.array(pd.read_csv(r0_path, sep=" ", header=None))
            if len(Rips0) and np.isinf(Rips0[-1, 1]):
                Rips0 = Rips0[:-1]

            Rips1 = np.array(pd.read_csv(r1_path, sep=" ", header=None))
            if len(Rips1) and np.isnan(Rips1[-1, 1]):
                Rips1[-1, 1] = 0

            return [[Rips0, Rips1], None, None]
    return []


def normalize_curve(curve, mode="l1"):
    if mode == "none":
        return curve
    if mode == "l1":
        s = np.sum(np.abs(curve))
        return curve / (s + 1e-12)
    raise ValueError("Unknown normalization mode.")


def apply_threshold(pairs, thr):
    if pairs is None or len(pairs) == 0:
        return np.empty((0, 2))
    pairs = np.asarray(pairs, float)
    if pairs.ndim != 2 or pairs.shape[1] < 2:
        return np.empty((0, 2))
    pers = pairs[:, 1] - pairs[:, 0]
    sel = (pers >= thr) & np.isfinite(pers)
    return pairs[sel]


def compute_global_thresholds(base, dim_intervals, thr_mode="0"):
    thr_dim = {}
    if thr_mode == "0":
        for d in dim_intervals:
            thr_dim[d] = 0.0
        return thr_dim

    for d in dim_intervals:
        pers_all = []
        for group in ["NonRelapse", "Relapse"]:
            root = os.path.join(base, group)
            if not os.path.isdir(root):
                continue
            for patient in sorted(os.listdir(root)):
                if patient.startswith("."):
                    continue
                p_dir = os.path.join(root, patient)
                for filename in sorted(os.listdir(p_dir)):
                    data = read_and_save(p_dir, filename)
                    if not data:
                        continue
                    diagrams = data[0]
                    pairs = diagrams[d]
                    if pairs is None or len(pairs) == 0:
                        break
                    pairs = np.asarray(pairs, float)
                    if pairs.ndim != 2 or pairs.shape[1] < 2:
                        break
                    pers = pairs[:, 1] - pairs[:, 0]
                    pers = pers[np.isfinite(pers)]
                    pers = pers[pers > 0]
                    if pers.size:
                        pers_all.append(pers)
                    break

        if pers_all:
            pers_cat = np.concatenate(pers_all)
            thr_dim[d] = float(np.percentile(pers_cat, 10))
        else:
            thr_dim[d] = 0.0

    return thr_dim


def compute_persistence_silhouette(diagrams, dimension, thr_dim):
    thr_value = thr_dim[dimension]
    pairs = diagrams[dimension]
    pairs = apply_threshold(pairs, thr_value)

    if pairs is None or len(pairs) == 0:
        return np.zeros(resolution)

    S = Silhouette(resolution=resolution)
    sil = S.fit_transform([pairs])[0]

    sil = normalize_curve(sil, normalization)

    if sigma > 0:
        sil = gaussian_filter1d(sil, sigma=sigma)

    return sil



thr_dim = compute_global_thresholds(base, dim_intervals, THR_MODE)

# NON RELAPSE
direct_NR = os.path.join(base, "NonRelapse")
listdirNR = [x for x in sorted(os.listdir(direct_NR)) if not x.startswith(".")]
SP_NonRelapse = []

for patient in listdirNR:
    listpac = sorted(os.listdir(os.path.join(direct_NR, patient)))
    Ripsaux = []
    for filename in listpac:
        data = read_and_save(os.path.join(direct_NR, patient), filename)
        if data:
            Ripsaux.append(data)
    if Ripsaux:
        diagrams = Ripsaux[0][0]
        sp0 = compute_persistence_silhouette(diagrams, 0, thr_dim)
        sp1 = compute_persistence_silhouette(diagrams, 1, thr_dim)
        feat = np.concatenate([sp0, sp1], axis=0)
        SP_NonRelapse.append(feat)

# RELAPSE
direct_R = os.path.join(base, "Relapse")
listdirR = [x for x in sorted(os.listdir(direct_R)) if not x.startswith(".")]
SP_Relapse = []

for patient in listdirR:
    listpac = sorted(os.listdir(os.path.join(direct_R, patient)))
    Ripsaux = []
    for filename in listpac:
        data = read_and_save(os.path.join(direct_R, patient), filename)
        if data:
            Ripsaux.append(data)
    if Ripsaux:
        diagrams = Ripsaux[0][0]
        sp0 = compute_persistence_silhouette(diagrams, 0, thr_dim)
        sp1 = compute_persistence_silhouette(diagrams, 1, thr_dim)
        feat = np.concatenate([sp0, sp1], axis=0)
        SP_Relapse.append(feat)

folder = "PSilhouette01"
subfolder = os.path.join(base, folder)
os.makedirs(os.path.join(subfolder, "Relapse"), exist_ok=True)
os.makedirs(os.path.join(subfolder, "NonRelapse"), exist_ok=True)

for i, curve in enumerate(SP_Relapse):
    np.savetxt(os.path.join(subfolder, "Relapse", f"{listdirR[i]}.csv"), curve)

for i, curve in enumerate(SP_NonRelapse):
    np.savetxt(os.path.join(subfolder, "NonRelapse", f"{listdirNR[i]}.csv"), curve)